In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import *

apply_plot_style()
FIGURES_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

In [ ]:
import time
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

raw = load_split("raw")
prep = load_split("prep")

print(f"raw  train {raw[0].shape}  test {raw[2].shape}  ({int(raw[0].isna().sum().sum())} missing)")
print(f"prep train {prep[0].shape}  test {prep[2].shape}  ({int(prep[0].isna().sum().sum())} missing)")
print(f"train positive rate {raw[1].mean():.3f}   test positive rate {raw[3].mean():.3f}")

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

SEARCH_SPACE = {
    "3_logistic_regression": {"C": [0.01, 0.1, 1, 10, 100]},
    "4_random_forest": {"n_estimators": [300, 600],
                        "max_depth": [None, 4, 8],
                        "min_samples_leaf": [1, 3]},
    "5_xgboost": {"max_depth": [2, 3, 5],
                  "learning_rate": [0.03, 0.1],
                  "n_estimators": [200, 500],
                  "subsample": [0.8, 1.0]},
}

rows = []
for name, grid in SEARCH_SPACE.items():
    n = int(np.prod([len(v) for v in grid.values()]))
    for param, values in grid.items():
        rows.append({"model": MODEL_LABELS[name], "hyperparameter": param,
                     "values searched": ", ".join(str(v) for v in values),
                     "candidates": n})
search_table = pd.DataFrame(rows)
search_table.to_csv(FIGURES_DIR / "table_6_1_search_space.csv", index=False)
search_table

In [ ]:
def run_tabfm(X_tr, y_tr, X_te):
    from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch as tabfm_v1_0_0
    clf = TabFMClassifier(model=tabfm_v1_0_0.load(model_type="classification"))
    clf.fit(X_tr, y_tr)
    return positive_proba(clf, X_te), None, clf, None


def run_search(estimator, grid, X_tr, y_tr, X_te):
    search = GridSearchCV(estimator, grid, scoring="roc_auc", cv=CV,
                          n_jobs=-1, return_train_score=True)
    search.fit(X_tr, y_tr)
    best = search.best_estimator_
    trace = pd.DataFrame(search.cv_results_)[
        ["params", "mean_test_score", "std_test_score", "mean_train_score", "rank_test_score"]]
    return positive_proba(best, X_te), search.best_params_, best, trace


def run_xgboost(X_tr, y_tr, X_te):
    from xgboost import XGBClassifier
    est = XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", tree_method="hist")
    return run_search(est, SEARCH_SPACE["5_xgboost"], X_tr, y_tr, X_te)


JOBS = [
    ("1_tabfm_raw", raw, run_tabfm),
    ("2_tabfm_preprocessed", prep, run_tabfm),
    ("3_logistic_regression", prep, lambda a, b, c: run_search(
        LogisticRegression(max_iter=5000, random_state=RANDOM_SEED),
        SEARCH_SPACE["3_logistic_regression"], a, b, c)),
    ("4_random_forest", prep, lambda a, b, c: run_search(
        RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1),
        SEARCH_SPACE["4_random_forest"], a, b, c)),
    ("5_xgboost", prep, run_xgboost),
]

In [ ]:
results, preds, traces = [], [], []

for name, data, fn in JOBS:
    X_tr, y_tr, X_te, y_te, ids = data
    print(f"{name} ... ", end="", flush=True)
    start = time.perf_counter()
    try:
        prob, params, fitted, trace = fn(X_tr, y_tr, X_te)
    except ImportError as e:
        print(f"skipped ({e.name} not installed)")
        continue
    elapsed = time.perf_counter() - start

    results.append({"model": name, **score(y_te, prob, elapsed)})
    preds.append(pd.DataFrame({"model": name, "id": ids, "y_true": y_te, "prob": prob}))
    if trace is not None:
        trace = trace.assign(model=name)
        traces.append(trace)
    if params is not None:
        joblib.dump(fitted, MODELS_DIR / f"{name}.joblib")
        joblib.dump(list(X_tr.columns), MODELS_DIR / f"{name}_features.joblib")

    print(f"roc_auc {results[-1]['roc_auc']:.3f}  in {elapsed:.1f}s"
          + (f"  best {params}" if params else ""))

In [ ]:
results = pd.DataFrame(results).set_index("model").round(4)
results.to_csv(FIGURES_DIR / "model_results.csv")
pd.concat(preds, ignore_index=True).to_csv(PROCESSED_DIR / "predictions.csv", index=False)

if traces:
    all_traces = pd.concat(traces, ignore_index=True)
    all_traces["params"] = all_traces["params"].astype(str)
    all_traces.to_csv(FIGURES_DIR / "tuning_trace.csv", index=False)
    best = (all_traces.sort_values("rank_test_score").groupby("model").first()
            .reset_index()[["model", "params", "mean_test_score", "std_test_score"]])
    best["model"] = best["model"].map(MODEL_LABELS)
    best.columns = ["Model", "Best configuration", "CV ROC-AUC", "SD"]
    best.round(4).to_csv(FIGURES_DIR / "table_6_3_best_config.csv", index=False)
    print(best.round(4).to_string(index=False))

print()
results

In [ ]:
full_prep = pd.read_csv(PROCESSED_DIR / "heart_disease_preprocessed.csv")
full_raw = pd.read_csv(PROCESSED_DIR / "heart_disease_raw.csv")
full_raw = full_raw[full_raw["id"].isin(full_prep["id"])]
sites = sorted(full_prep[SITE_COL].unique())

BEST = {}
for name in SEARCH_SPACE:
    path = MODELS_DIR / f"{name}.joblib"
    if path.exists():
        BEST[name] = joblib.load(path).get_params()

def frames(name, held):
    src = full_raw if name == "1_tabfm_raw" else full_prep
    tr = src[src[SITE_COL] != held]
    te = src[src[SITE_COL] == held]
    drop = [c for c in NON_FEATURES if c in src.columns]
    return (tr.drop(columns=drop), tr[TARGET].to_numpy(),
            te.drop(columns=drop), te[TARGET].to_numpy())

_TABFM_CACHE = {}

def fit_predict(name, X_tr, y_tr, X_te):
    if name.startswith(("1_tabfm", "2_tabfm")):
        # Load the weights once and reuse them across folds. Reloading per fold
        # costs roughly a minute each and dominates the runtime of this cell.
        from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch as tabfm_v1_0_0
        if "model" not in _TABFM_CACHE:
            _TABFM_CACHE["model"] = tabfm_v1_0_0.load(model_type="classification")
        clf = TabFMClassifier(model=_TABFM_CACHE["model"])
        clf.fit(X_tr, y_tr)
        return positive_proba(clf, X_te)
    if name == "3_logistic_regression":
        est = LogisticRegression(**BEST[name])
    elif name == "4_random_forest":
        est = RandomForestClassifier(**BEST[name])
    else:
        from xgboost import XGBClassifier
        est = XGBClassifier(**BEST[name])
    return positive_proba(est.fit(X_tr, y_tr), X_te)

loho_rows = []
for name, _, _ in JOBS:
    if name in SEARCH_SPACE and name not in BEST:
        continue
    for held in sites:
        X_tr, y_tr, X_te, y_te = frames(name, held)
        try:
            prob = fit_predict(name, X_tr, y_tr, X_te)
        except ImportError:
            break
        loho_rows.append({"model": name, "held_out": held, "n_train": len(X_tr),
                          "n_test": len(X_te), "test_prevalence": round(y_te.mean(), 3),
                          "roc_auc": round(score(y_te, prob)["roc_auc"], 4)})
        print(f"{name:<22} hold out {held:<11} roc_auc {loho_rows[-1]['roc_auc']:.3f}")

loho = pd.DataFrame(loho_rows)
loho.to_csv(FIGURES_DIR / "loho_results.csv", index=False)

wide = loho.pivot(index="model", columns="held_out", values="roc_auc")
wide["mean"] = wide.mean(axis=1).round(3)
wide["worst"] = wide[sites].min(axis=1)
wide["drop_vs_random_split"] = (
    results["roc_auc"].reindex(wide.index) - wide["mean"]).round(3)
wide.rename(index=MODEL_LABELS).to_csv(FIGURES_DIR / "table_7_4_loho.csv")
print()
wide.rename(index=MODEL_LABELS)